# 06 — Exportación a `data/processed/`
Cada tabla en CSV y en Parquet, más los archivos de auditoría y el diccionario de datos.
El Parquet es la copia de referencia porque conserva los tipos (entero con nulos, booleano);
el CSV los pierde al releerse.

In [ ]:
%run -i modulos/comun.ipynb
%run -i modulos/diccionario.ipynb
import shutil
import matplotlib.pyplot as plt

# archivo normalizado -> tabla final
SALIDAS = {
    "norm_causas_movimiento.csv": "causas_movimiento",
    "norm_causas_gestiones.csv": "causas_por_gestion",
    "norm_causas_historico.csv": "causas_serie_historica",
    "norm_juzgados.csv": "juzgados",
    "norm_personal.csv": "personal",
    "norm_personal_jurisdiccional.csv": "personal_jurisdiccional",
    "norm_sumariante.csv": "autoridad_sumariante",
    "norm_procesos_causas.csv": "causas_por_tipo_proceso",
    "norm_procesos_resueltas.csv": "resueltas_por_tipo_proceso",
    "norm_procesos_apelacion.csv": "apelaciones_por_tipo_proceso",
    "norm_procesos_ejecucion.csv": "ejecucion_por_tipo_proceso",
    "norm_procesos_otros.csv": "otros_tramites_por_tipo_proceso",
}

AUDITORIA = ["discrepancias.csv", "catalogo_cuadros.csv", "problemas_extraccion.csv", "problemas_normalizacion.csv",
             "equivalencias_candidatas.csv", "columnas_4_1_encabezados.csv", "filas_complementarias.csv",
             "problemas_extraccion_procesos.csv", "firmas_procesos.csv", "validacion_geografia.csv",
             "inconsistencias_geografia.csv", "validacion_juzgados.csv", "validacion_contexto_procesos.csv",
             "validacion_correcciones_tipo_proceso.csv"]

# Columnas enteras aunque tengan nulos.
ENTERAS = {"pagina_pdf", "gestion", "num_juzgados", "pendientes_inicio", "ingresadas", "atendidas", "resueltas",
           "pendientes_fin", "items_mujer", "items_varon", "items_acefalias", "items_total", "remun_mujer",
           "remun_varon", "remun_acefalias", "remun_total", "amonestacion", "multa", "suspension", "destitucion",
           "total_sanciones", "recibidas", "rechazadas", "en_tramite", "resoluciones_primera_instancia", "valor",
           "items", "remuneracion", "fila_en_cuadro", "n_columnas", "num_juzgados_pagina", "orden_fila",
           "orden_columna", "nuevas_ingresadas", "readecuadas_ley_439", "recibidas_excusa_recusacion",
           "preliminares_formalizados", "cautelares_formalizados", "nivel_jerarquia"}
BOOLEANAS = {"revisado_manual", "errata_corregida", "es_ultima_columna", "distrito_derivado_del_bloque",
             "rotulo_1_derivado_del_bloque", "es_hoja_jerarquia"}


def tipos_finales(df):
    for col in df.columns:
        if col in ENTERAS:
            df[col] = pd.to_numeric(df[col], errors="coerce").astype("Int64")
        elif col.startswith("pct_") or col.startswith("col_sin_rotulo") or col.endswith("_pct") or col == "promedio_por_juzgado":
            df[col] = pd.to_numeric(df[col], errors="coerce").astype("Float64")
        elif col in BOOLEANAS:
            df[col] = df[col].astype("boolean")
    return df

## Las doce tablas

In [ ]:
PROCESSED.mkdir(parents=True, exist_ok=True)
tablas = {}
for archivo in SALIDAS:
    nombre = SALIDAS[archivo]
    df = tipos_finales(pd.read_csv(INTERIM / archivo, low_memory=False))
    df.to_csv(PROCESSED / (nombre + ".csv"), index=False, encoding="utf-8")
    df.to_parquet(PROCESSED / (nombre + ".parquet"), index=False)
    tablas[nombre] = df
    print(nombre, len(df), "filas", len(df.columns), "columnas")

auditoria = PROCESSED / "auditoria"
auditoria.mkdir(exist_ok=True)
for archivo in AUDITORIA:
    shutil.copy(INTERIM / archivo, auditoria / archivo)

In [ ]:
filas_por_tabla = []
for nombre in tablas:
    filas_por_tabla.append((nombre, len(tablas[nombre])))
filas_por_tabla.sort(key=lambda t: t[1])
plt.figure(figsize=(10, 5))
plt.barh([t[0] for t in filas_por_tabla], [t[1] for t in filas_por_tabla], color="tab:blue")
plt.xscale("log")
plt.xlabel("filas (escala log)")
plt.title("Las doce tablas exportadas")
plt.tight_layout()
plt.show()

## Toda columna exportada tiene que estar descrita en el módulo diccionario

In [ ]:
huerfanas = []
for tabla in tablas:
    if tabla not in TABLAS:
        huerfanas.append("tabla " + tabla + " sin entrada en TABLAS")
    for col in tablas[tabla].columns:
        if (tabla, col) not in COLUMNAS_POR_TABLA and col not in COLUMNAS:
            huerfanas.append(tabla + "." + col)
if len(huerfanas) > 0:
    raise RuntimeError("Columnas exportadas sin documentar en el módulo diccionario: " + ", ".join(huerfanas))
print("Todas las columnas están documentadas.")

## Diccionario de columnas y de valores

In [ ]:
def ficha(tabla, columna):
    f = COLUMNAS_POR_TABLA.get((tabla, columna))
    if not f:
        f = COLUMNAS[columna]
    return f


filas = []
for tabla in tablas:
    df = tablas[tabla]
    for col in df.columns:
        f = ficha(tabla, col)
        serie = df[col]
        no_nulos = int(serie.notna().sum())
        numerica = pd.api.types.is_numeric_dtype(serie) and no_nulos > 0
        ejemplo = ""
        for x in serie:
            if pd.notna(x):
                ejemplo = str(x)
                break
        filas.append({
            "tabla": tabla, "columna": col, "tipo": str(serie.dtype), "unidad": f["unidad"], "origen": f["origen"],
            "descripcion": " ".join(f["desc"].split()), "no_nulos": no_nulos, "nulos": len(df) - no_nulos,
            "valores_distintos": int(serie.nunique(dropna=True)),
            "minimo": serie.min() if numerica else "",
            "maximo": serie.max() if numerica else "",
            "ejemplo": ejemplo,
        })
columnas_dic = pd.DataFrame(filas)
columnas_dic.to_csv(PROCESSED / "diccionario_de_datos.csv", index=False, encoding="utf-8")
print("diccionario_de_datos.csv:", len(columnas_dic), "columnas documentadas")

In [ ]:
def titulo_limpio(titulo):
    # El título del paso 02 arrastra fragmentos de la grilla: se deja el título y las líneas "Por:" y "Según:".
    partes = []
    for p in str(titulo).split(" / "):
        if p.strip():
            partes.append(p.strip())
    if len(partes) == 0:
        return ""
    limpio = [partes[0]]
    for p in partes[1:]:
        if p.lower().startswith(("por:", "según:", "segun:")):
            limpio.append(p)
    return " / ".join(limpio)


filas = []
for tabla in tablas:
    df = tablas[tabla]
    for col in VALORES:
        if col not in df.columns:
            continue
        for valor, frecuencia in df[col].value_counts(dropna=True).items():
            filas.append({"tabla": tabla, "columna": col, "valor": valor, "frecuencia": int(frecuencia),
                          "significado": VALORES[col].get(valor, "SIN DOCUMENTAR: valor no previsto en diccionario.VALORES")})

cat = pd.read_csv(INTERIM / "catalogo_cuadros.csv", dtype=str).fillna("")
titulos = {}
for i, r in cat[cat["tipo"] == "cuadro"].iterrows():
    if r["cuadro_id"] not in titulos:
        titulos[r["cuadro_id"]] = (r["rango_paginas"], titulo_limpio(r["titulo"]))
for tabla in tablas:
    df = tablas[tabla]
    if "cuadro_origen" not in df.columns:
        continue
    for valor, frecuencia in df["cuadro_origen"].value_counts().items():
        pagina, titulo = titulos.get(valor, ("", ""))
        if titulo:
            significado = "p. " + pagina + " — " + titulo
        else:
            significado = "Tabla sin número de cuadro en el anuario."
        filas.append({"tabla": tabla, "columna": "cuadro_origen", "valor": valor, "frecuencia": int(frecuencia),
                      "significado": significado})
valores_dic = pd.DataFrame(filas)
valores_dic.to_csv(PROCESSED / "diccionario_de_valores.csv", index=False, encoding="utf-8")
print("diccionario_de_valores.csv:", len(valores_dic), "valores documentados")

sin_documentar = valores_dic[valores_dic["significado"].str.startswith("SIN DOCUMENTAR")]
sin_documentar